In [ ]:
import os
print(os.getpid())

In [ ]:
from cobra.io import load_model, load_json_model
from cobra.flux_analysis import pfba

import time
import cProfile
import pstats

In [ ]:
# Contains the main steps of the BayesOpt
%run BayesOpt_MOBO_comprehensive.ipynb

In [ ]:
# Plotting functions to be used across notebooks
%run Plotting_MOBO_comprehensive.ipynb

# iML1515

### Model & Medium

In [ ]:
# load iML1515
model_iML1515 = load_model("iML1515")
medium_iML1515 = model_iML1515.medium

In [ ]:
# M9 with essential trace metals
medium_iJO1366_reduced = {
    'EX_pi_e': 34.90, # in M9
    'EX_mn2_e': 0.001, # - required?; drops at 0.0001
    'EX_fe2_e': 0.1, # - required?; drops at 0.01
    'EX_glc__D_e': 10.0, # in M9
    'EX_zn2_e': 0.001, # - required?; drops at 0.0001
    'EX_mg2_e': 1.0, # in M9 
    'EX_ca2_e': 0.05, # in M9
    'EX_ni2_e': 0.001, # - required?; drops at 0.0001
    'EX_cu2_e': 0.001, # - required?; drops at 0.0001
    'EX_cobalt2_e': 0.0001, # - required; drops at 0.00001 
    'EX_mobd_e': 0.0005, # - required?; drops at 0.000001
    'EX_so4_e': 1.0, # in M9
    'EX_nh4_e': 9.3475, # in M9
    'EX_k_e': 11.02, # in M9
    'EX_na1_e': 52.038, # in M9
    'EX_cl_e': 13.6755, # in M9
    'EX_o2_e': 20.0, # in M9 II - drops at 10
}
bounds_iJO1366_reduced = { # 10 decision variables
    'EX_pi_e': (0.0, 50),
    'EX_mn2_e': (0.001, 0.001), # fix "trace" 
    'EX_fe2_e': (0.1, 0.1), # fix "trace"
    'EX_glc__D_e': (1.0, 10),
    'EX_zn2_e': (0.001, 0.001), # fix "trace"
    'EX_mg2_e': (0.0, 10),
    'EX_ca2_e': (0.0, 10),
    'EX_ni2_e': (0.001, 0.001), # fix "trace"
    'EX_cu2_e': (0.001, 0.001), # fix "trace"
    'EX_cobalt2_e': (0.0001, 0.0001), # fix "trace"
    'EX_mobd_e': (0.0005, 0.0005), # fix "trace"
    'EX_so4_e': (0.0, 10),
    'EX_nh4_e': (0.0, 10),
    'EX_k_e': (0.0, 20),
    'EX_na1_e': (0.0, 100.0), # if fixed, can be set to 0
    'EX_cl_e': (0.0, 20),
    'EX_o2_e': (0, 20), # upper bound can't be set to 10 or lower
}
# costs are in £/mol
costs_iJO1366_reduced = {
    'EX_pi_e': 23.4234, # Phosphate - approximate (several sources)
    'EX_mn2_e': 0.0, #33.25, # Manganese - MnCl2·4H20
    'EX_fe2_e': 0.0, #37.5, # IronII - as iron sulfate FeSO4·7H2O
    'EX_glc__D_e': 7.7647236, # Glucose
    'EX_zn2_e': 0.0, #28.3, # Zinc - as Zn(CH3CHOOH)·H2O
    'EX_mg2_e': 19.1022, # Magnesium - as MgSO4·7H2O - approximate bc. half of 38.2044
    'EX_ca2_e': 18.08223, # Calcium - as CaCl2·2H2O
    'EX_ni2_e': 0.0, #53.24, # Nickel - as NiCl2·6H2O
    'EX_cu2_e': 0.0, #31.37, # Copper - CuCl2·2H2O
    'EX_cobalt2_e': 0.0, #114.39, # Cobalt - as CoCl2·6H2O
    'EX_mobd_e': 0.0, #184.12, # Molybdenum - molybdate NaMoO4·2H2O
    'EX_so4_e': 19.1022, # Sulfate - as MgSO4·7H2O - approximate bc. half of 38.2044
    'EX_nh4_e': 3.6748, # Ammonia - as NH4Cl - approximate (several sources and "side-effect"
    'EX_k_e': 20.82177, # Potassium - as KCl - approximate (several sources and "side-effect"
    'EX_na1_e': 0.0, # Sodium - as NaCl, Na2HPO4
    'EX_cl_e': 3.03888, # Chlorid - as NaCl, NH4Cl, CaCl2 - approximate price from NaCl
    'EX_o2_e': 0.0, # oxygen - no costs
}


In [ ]:
"""
NOTES for constraints

- [Mg] = [SO4]; bounds are the same
    - Mg has index 5 in medium dictionary
    - SO4 has index 11
    - the have the same upper bound

- [Cl] = 2[Ca] + [NH4] + ([Na]-2([PO4]-[K]))
    -> [Cl] - [NH4] - 2[Ca] - [Na] + 2[PO4] - 2[K] = 0
    - indices
        Cl: 15, NH4: 12, Ca: 6, Na: 14, PO4: 0, K: 13
    - upper bounds
        20 - 10 - 2*10 - 100 + 2*50 - 2*20 = 0
        -> 2 - 1 - 2 - 10 + 10 - 4 = 0
    
"""
medium_equality_constraints = [
    # [Mg] = [SO4]
    (
        torch.tensor([5, 11]),
        torch.tensor([1.0, -1.0], dtype = torch.double),
        0.0
    ),

    # Chloride balance
    (
        torch.tensor([15, 12, 6, 14, 0, 13]),
        torch.tensor([2.0, -1.0, -2.0, -10.0, 10.0, -4.0], dtype = torch.double),
        0.0
    )
]

## Optimisation

In [ ]:
# set n_iter and date to be used in all calls and names
date = "2026-03-09"
n_start = 10 # how many random media compositions to initialise the algorithm
n_iter = 0 # how many iterations per run
iterations = str(n_iter)
n_candidates = 5 # batch size

biomass_rxn_id = "BIOMASS_Ec_iML1515_core_75p37M"

# set medium
medium = medium_iJO1366_reduced
bounds = bounds_iJO1366_reduced
costs = costs_iJO1366_reduced
# setting the upper limit of oxygen to 20, reduces the maximum growth rate to 0.822
model_iML1515.medium = medium

"""performance with chosen medium"""
fba_sol = model_iML1515.optimize()
print(fba_sol.status)
pfba_solution = pfba(model_iML1515)
# print relevant fluxes of optimised flux distribution
print("Biomass flux:\t", pfba_solution.fluxes["BIOMASS_Ec_iML1515_core_75p37M"])

# bounds of biomass reaction
biomass_rxn = model_iML1515.reactions.get_by_id("BIOMASS_Ec_iML1515_core_75p37M")
biomass_rxn.bounds = (0.0, 0.85)
# define objective as combination of growth (biomass) and production
factor_bio = 1
combined_objective = model_iML1515.problem.Objective(
    factor_bio * model_iML1515.reactions.BIOMASS_Ec_iML1515_core_75p37M.flux_expression,
    direction = 'max')
model_iML1515.objective = combined_objective
print(model_iML1515.objective)
pfba_solution = pfba(model_iML1515)
# print relevant fluxes of optimised flux distribution
print("Biomass flux:\t", pfba_solution.fluxes["BIOMASS_Ec_iML1515_core_75p37M"])

### Growth-Cost

In [ ]:
opt_objective = "growth-cost"

start_time = time.time() # when did the algorithm start

profiler1 = cProfile.Profile()
profiler1.enable()

results_iML1515_M9 = media_BayesOpt(
    MetModel = model_iML1515,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = None,
    n_start = n_start,
    data_start = None,
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = combined_objective,
    start_time = start_time,
    medium_linear_equality_constraints = medium_equality_constraints,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = True
    )

profiler1.disable()
profiler1.dump_stats("profile_iML1515.prof")

# print the 30 most expensive functions
stats1 = pstats.Stats(profiler1)
stats1.sort_stats("cumtime").print_stats(30)

# Display runtime
print_runtime(start_time)

# plot & save results
basename = (date + "_BayesOpt_pfba_iML1515_" + opt_objective + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iML1515_M9, basename)

plot_pareto_batch_colour(
    results = results_iML1515_M9,
    figname = (basename + "_pareto_batch_colour.png"),
    MetModel = model_iML1515,
    initial_medium = medium,
    initial_costs = costs
)
plot_growth_per_cost(
    results = results_iML1515_M9,
    figname = (basename + "_growth-per-cost.png")
)

# iJO1366

## Model & Medium

In [ ]:
# load modified iJO1366
model_iJO1366_antiEpEX_scFv = load_json_model("..\\iJO1366_producing_antiEpEX-scFv.json")
medium_iJO1366 = model_iJO1366_antiEpEX_scFv.medium
print(model_iJO1366_antiEpEX_scFv)

In [ ]:
# M9 with essential trace metals and amino acids
medium_iJO1366_enriched = {
    'EX_pi_e': 34.90, # in M9
    'EX_mn2_e': 0.001, # - required?; drops at 0.0001
    'EX_fe2_e': 0.1, # - required?; drops at 0.01
    'EX_glc__D_e': 10.0, # in M9
    'EX_zn2_e': 0.001, # - required?; drops at 0.0001
    'EX_mg2_e': 1.0, # in M9 
    'EX_ca2_e': 0.05, # in M9
    'EX_ni2_e': 0.001, # - required?; drops at 0.0001
    'EX_cu2_e': 0.001, # - required?; drops at 0.0001
    'EX_cobalt2_e': 0.0001, # - required; drops at 0.00001 
    'EX_mobd_e': 0.0005, # - required?; drops at 0.0001 - higher than for iML1515
    'EX_so4_e': 1.0, # in M9
    'EX_nh4_e': 9.3475, # in M9 # lower than with 10
    'EX_k_e': 11.02, # in M9
    'EX_na1_e': 52.038, # in M9
    'EX_cl_e': 13.6755, # in M9
    'EX_o2_e': 20.0, # in M9 II - drops at 10
    'EX_arg__L_e': 4.75, # L-Arginine
    'EX_asn__L_e': 3.05, # L-Asparagine
    'EX_gln__L_e' : 4.95 # L-Glutamine
}
bounds_iJO1366_enriched = { # 12 decision variables
    'EX_pi_e': (0.0, 50),
    'EX_mn2_e': (0.001, 0.001), # fixed "trace" 
    'EX_fe2_e': (0.1, 0.1), # fixed "trace"
    'EX_glc__D_e': (1.0, 10),
    'EX_zn2_e': (0.001, 0.001), # fixed "trace"
    'EX_mg2_e': (0.0, 10),
    'EX_ca2_e': (0.0, 10),
    'EX_ni2_e': (0.001, 0.001), # fixed "trace"
    'EX_cu2_e': (0.001, 0.001), # fixed "trace"
    'EX_cobalt2_e': (0.0001, 0.0001), # fixed "trace"
    'EX_mobd_e': (0.0005, 0.0005), # fixed "trace"
    'EX_so4_e': (0.0, 10),
    'EX_nh4_e': (0.0, 10), # close to the default of 9.3475
    'EX_k_e': (0.0, 20),
    'EX_na1_e': (0.0, 100.0), # if fixed, can be set to 0
    'EX_cl_e': (0.0, 20),
    'EX_o2_e': (0, 20), # if fixed - can't be set to 10 or lower
    'EX_arg__L_e': (0.0, 10.0), # L-Arginine
    'EX_asn__L_e': (0.0, 10.0), # L-Asparagine
    'EX_gln__L_e' : (0.0, 10.0) # L-Glutamine
}
# costs are in £/mol
costs_iJO1366_enriched = {
    'EX_pi_e': 23.4234, # Phosphate - approximate (several sources)
    'EX_mn2_e': 0.0, #33.25, # Manganese - MnCl2·4H20
    'EX_fe2_e': 0.0, #37.5, # IronII - as iron sulfate FeSO4·7H2O
    'EX_glc__D_e': 7.7647236, # Glucose
    'EX_zn2_e': 0.0, #28.3, # Zinc - as Zn(CH3CHOOH)·H2O
    'EX_mg2_e': 19.1022, # Magnesium - as MgSO4·7H2O - approximate bc. half of 38.2044
    'EX_ca2_e': 18.08223, # Calcium - as CaCl2·2H2O
    'EX_ni2_e': 0.0, #53.24, # Nickel - as NiCl2·6H2O
    'EX_cu2_e': 0.0, #31.37, # Copper - CuCl2·2H2O
    'EX_cobalt2_e': 0.0, #114.39, # Cobalt - as CoCl2·6H2O
    'EX_mobd_e': 0.0, #184.12, # Molybdenum - molybdate NaMoO4·2H2O
    'EX_so4_e': 19.1022, # Sulfate - as MgSO4·7H2O - approximate bc. half of 38.2044
    'EX_nh4_e': 3.6748, # Ammonia - as NH4Cl - approximate (several sources and "side-effect"
    'EX_k_e': 20.82177, # Potassium - as KCl - approximate (several sources and "side-effect"
    'EX_na1_e': 0.0, # Sodium - as NaCl, Na2HPO4
    'EX_cl_e': 3.03888, # Chlorid - as NaCl, NH4Cl, CaCl2 - approximate price from NaCl
    'EX_o2_e': 0.0, # oxygen - no costs
    'EX_arg__L_e': 61.1442, # L-Arginine
    'EX_asn__L_e': 93.01248, # L-Asparagine
    'EX_gln__L_e' : 80.23086 # L-Glutamine
}

In [ ]:
# extract biomass and production reaction ids
protein_rxn = model_iJO1366_antiEpEX_scFv.reactions.get_by_id("Recombinant_protein")
biomass_rxn = model_iJO1366_antiEpEX_scFv.reactions.get_by_id("BIOMASS_Ec_iJO1366_core_53p95M")
biomass_rxn_bounds = biomass_rxn.bounds # default bounds of (0, 1000)
# limit growth rate to 0.85
biomass_rxn.bounds = (0.0, 0.85)


# create combined objective
factor_bio = 0.99 # when set to 1, only biomass production is optimised, when smaller, protein production included
factor_prot = 1 - factor_bio
combined_objective = model_iJO1366_antiEpEX_scFv.problem.Objective(
    factor_bio * model_iJO1366_antiEpEX_scFv.reactions.BIOMASS_Ec_iJO1366_core_53p95M.flux_expression + 
    factor_prot * model_iJO1366_antiEpEX_scFv.reactions.Recombinant_protein.flux_expression,
    direction = 'max')
# Set model objective to combined objective
model_iJO1366_antiEpEX_scFv.objective = combined_objective
print("Objective:\n", model_iJO1366_antiEpEX_scFv.objective)

In [ ]:
"""
NOTES for constraints

- [Mg] = [SO4]; bounds are the same
    - Mg has index 5 in medium dictionary
    - SO4 has index 11
    - the have the same upper bound

- [Cl] = 2[Ca] + [NH4] + ([Na]-2([PO4]-[K]))
    -> [Cl] - [NH4] - 2[Ca] - [Na] + 2[PO4] - 2[K] = 0
    - indices
        Cl: 15, NH4: 12, Ca: 6, Na: 14, PO4: 0, K: 13
    - upper bounds
        20 - 10 - 2*10 - 100 + 2*50 - 2*20 = 0
        -> 2 - 1 - 2 - 10 + 10 - 4 = 0
    
"""
medium_equality_constraints = [
    # [Mg] = [SO4]
    (
        torch.tensor([5, 11]),
        torch.tensor([1.0, -1.0], dtype = torch.double),
        0.0
    ),

    # Chloride balance
    (
        torch.tensor([15, 12, 6, 14, 0, 13]),
        torch.tensor([2.0, -1.0, -2.0, -10.0, 10.0, -4.0], dtype = torch.double),
        0.0
    )
]

## Optimisations

In [ ]:
# set n_iter and date to be used in all calls and names
date = "2026-03-09"
n_start = 10 # how many random media compositions to initialise the algorithm
n_iter = 5 # how many media compositions to evaluate; saw convergence after about 40
iterations = str(n_iter)
n_candidates = 5

medium = medium_iJO1366_enriched
bounds = bounds_iJO1366_enriched
costs = costs_iJO1366_enriched

biomass_rxn_id = "BIOMASS_Ec_iJO1366_core_53p95M"
protein_rxn_id = "Recombinant_protein"

# bounds of biomass reaction
biomass_rxn.bounds = (0.0, 0.85)
# define objective as combination of growth (biomass) and production
factor_bio = 0.99
factor_prot = 1-factor_bio
combined_objective = model_iJO1366_antiEpEX_scFv.problem.Objective(
    factor_bio * model_iJO1366_antiEpEX_scFv.reactions.BIOMASS_Ec_iJO1366_core_53p95M.flux_expression + 
    factor_prot * model_iJO1366_antiEpEX_scFv.reactions.Recombinant_protein.flux_expression,
    direction = 'max')
model_iJO1366_antiEpEX_scFv.objective = combined_objective
print(model_iJO1366_antiEpEX_scFv.objective)

### Growth-Cost

In [ ]:
opt_objective = "growth-cost"

start_time = time.time() # when did the algorithm start

profiler2 = cProfile.Profile()
profiler2.enable()

results_iJO1366_gc = media_BayesOpt(
    MetModel = model_iJO1366_antiEpEX_scFv,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = combined_objective,
    start_time = start_time,
    medium_linear_equality_constraints = None, #medium_equality_constraints,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
    )

profiler2.disable()
profiler2.dump_stats("profile_iJO1366_gc.prof")

# print the 30 most expensive functions
stats2 = pstats.Stats(profiler2)
stats2.sort_stats("cumtime").print_stats(30)

# Display runtime
print_runtime(start_time)

basename = (date + "_BayesOpt_iJO1366_antiEpEX_scFv_" + opt_objective + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iJO1366_gc, basename)

# plot
plot_growth_per_cost(results_iJO1366_gc, (basename + "_growth-per-cost.png"))
plot_pareto_batch_colour(
    results_iJO1366_gc,
    figname = (basename + "_pareto.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )   

### Growth-Production

In [ ]:
opt_objective = "growth-production"
start_time = time.time() # when did the algorithm start

profiler3 = cProfile.Profile()
profiler3.enable()

results_iJO1366_gp = media_BayesOpt(
    MetModel = model_iJO1366_antiEpEX_scFv,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = combined_objective,
    start_time = start_time,
    medium_linear_equality_constraints = None, #medium_equality_constraints,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
    )

profiler3.disable()
profiler3.dump_stats("profile_iJO1366_gp.prof")

# print the 30 most expensive functions
stats3 = pstats.Stats(profiler3)
stats3.sort_stats("cumtime").print_stats(30)

# Display runtime
print_runtime(start_time)

basename = (date + "_BayesOpt_iJO1366_antiEpEX_scFv_" + opt_objective + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iJO1366_gp, basename)

# plot
plot_pareto_batch_colour(
    results_iJO1366_gp,
    xax = "growth rate tensors", 
    yax = "cost tensors",
    figname = (basename + "_pareto_growth-cost.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
plot_pareto_batch_colour(
    results_iJO1366_gp,
    xax = "growth rate tensors", 
    yax = "production tensors",
    figname = (basename + "_pareto_growth-production.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )

### Production-Cost

In [ ]:
opt_objective = "production-cost"

start_time = time.time() # when did the algorithm start

profiler4 = cProfile.Profile()
profiler4.enable()

results_iJO1366_pc = media_BayesOpt(
    MetModel = model_iJO1366_antiEpEX_scFv,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = combined_objective,
    start_time = start_time,
    medium_linear_equality_constraints = None, #medium_equality_constraints,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
    )

profiler4.disable()
profiler4.dump_stats("profile_iJO1366_pc.prof")

# print the 30 most expensive functions
stats4 = pstats.Stats(profiler4)
stats4.sort_stats("cumtime").print_stats(30)

# Display runtime
print_runtime(start_time)

# plot & save results
basename = (date + "_BayesOpt_iJO1366_antiEpEX_scFv_" + opt_objective + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iJO1366_pc, basename)

# plot
plot_pareto_batch_colour(
    results_iJO1366_pc,
    xax = "production tensors", 
    yax = "cost tensors",
    figname = (basename + "_pareto_production-cost.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
plot_pareto_batch_colour(
    results_iJO1366_pc,
    xax = "growth rate tensors", 
    yax = "production tensors",
    figname = (basename + "_pareto_growth-production.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
plot_pareto_batch_colour(
    results_iJO1366_pc,
    xax = "growth rate tensors", 
    yax = "cost tensors",
    figname = (basename + "_pareto_growth-cost.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )

### Growth-Production-Cost

In [ ]:
opt_objective = "growth-production-cost"

start_time = time.time() # when did the algorithm start

profiler5 = cProfile.Profile()
profiler5.enable()

results_iJO1366_gpc = media_BayesOpt(
    MetModel = model_iJO1366_antiEpEX_scFv,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = combined_objective,
    start_time = start_time,
    medium_linear_equality_constraints = None, #medium_equality_constraints,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
    )

profiler5.disable()
profiler5.dump_stats("profile_iJO1366_gpc.prof")

# print the 30 most expensive functions
stats5 = pstats.Stats(profiler5)
stats5.sort_stats("cumtime").print_stats(30)

# Display runtime
print_runtime(start_time)

# plot & save results
basename = (date + "_BayesOpt_iJO1366_antiEpEX_scFv_" + opt_objective + "_" + iterations + "_pfba_constrained")

# store results in JSON file
JSON_serialize_store_results(results_iJO1366_gpc, basename)

# plot all results
plot_pareto_batch_colour(
    results_iJO1366_gpc,
    xax = "growth rate tensors", 
    yax = "cost tensors",
    figname = (basename + "_pareto_growth-cost.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
plot_pareto_batch_colour(
    results_iJO1366_gpc,
    xax = "growth rate tensors", 
    yax = "production tensors",
    figname = (basename + "_pareto_growth-production.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
    
plot_3D(
    results_iJO1366_gpc,
    xax = "growth rate tensors", 
    yax = "cost tensors", 
    zax = "production tensors",
    figname = (basename + "_3D.png"),
    MetModel = model_iJO1366_antiEpEX_scFv, 
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )

# plot subset of results where production > 0.01 and growth > 0.5
plot_production_per_cost_coloured_by_growth(
    results_iJO1366_gpc,
    figname = (basename + "_best_coloured-by-growth.png"),
    growth_threshold = 0.5,
    production_threshold = 0.01,
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
# plot subset of results where production > 0.01 and growth doesn't matter
plot_production_per_cost_coloured_by_growth(
    results_iJO1366_gpc,
    figname = (basename + "_best-production_coloured-by-growth.png"),
    growth_threshold = 0.0,
    production_threshold = 0.01,
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )

# How to load and deserialise the results
#result = JSON_deserialize_load_results((basename + ".json"), model_iJO1366_antiEpEX_scFv)

### Growth-Production-Cost - fixed M9

In [ ]:
# M9 with essential trace metals and amino acids - only aa variable
bounds_iJO1366_enriched_M9fix = {
    'EX_pi_e': (34.90, 34.90),
    'EX_mn2_e': (0.001, 0.001), 
    'EX_fe2_e': (0.1, 0.1),
    'EX_glc__D_e': (10, 10),
    'EX_zn2_e': (0.001, 0.001), 
    'EX_mg2_e': (1.0, 1.0),
    'EX_ca2_e': (0.05, 0.05),
    'EX_ni2_e': (0.001, 0.001),
    'EX_cu2_e': (0.001, 0.001), 
    'EX_cobalt2_e': (0.0001, 0.0001), 
    'EX_mobd_e': (0.0005, 0.0005), 
    'EX_so4_e': (1.0, 1.0),
    'EX_nh4_e': (9.3475, 9.3475), 
    'EX_k_e': (11.02, 11.02),
    'EX_na1_e': (52.038, 52.038), 
    'EX_cl_e': (13.6755,13.6755),
    'EX_o2_e': (20, 20),
    'EX_arg__L_e': (0.0, 10.0), # L-Arginine
    'EX_asn__L_e': (0.0, 10.0), # L-Asparagine
    'EX_gln__L_e' : (0.0, 10.0) # L-Glutamine
}

In [ ]:
# set n_iter and date to be used in all calls and names
date = "2026-03-09"
n_start = 10 # how many random media compositions to initialise the algorithm
n_iter = 5 # how many media compositions to evaluate; saw convergence after about 40
iterations = str(n_iter)
n_candidates = 5

medium = medium_iJO1366_enriched
bounds = bounds_iJO1366_enriched_M9fix
costs = costs_iJO1366_enriched

biomass_rxn_id = "BIOMASS_Ec_iJO1366_core_53p95M"
protein_rxn_id = "Recombinant_protein"

# bounds of biomass reaction
biomass_rxn.bounds = (0.0, 0.85)
# define objective as combination of growth (biomass) and production
factor_bio = 0.99
factor_prot = 1-factor_bio
combined_objective = model_iJO1366_antiEpEX_scFv.problem.Objective(
    factor_bio * model_iJO1366_antiEpEX_scFv.reactions.BIOMASS_Ec_iJO1366_core_53p95M.flux_expression + 
    factor_prot * model_iJO1366_antiEpEX_scFv.reactions.Recombinant_protein.flux_expression,
    direction = 'max')
model_iJO1366_antiEpEX_scFv.objective = combined_objective
print(model_iJO1366_antiEpEX_scFv.objective)
solution = model_iJO1366_antiEpEX_scFv.optimize()
solution.fluxes[biomass_rxn_id]

In [ ]:
opt_objective = "growth-production-cost"

start_time = time.time() # when did the algorithm start

profiler6 = cProfile.Profile()
profiler6.enable()

results_iJO1366_gpc = media_BayesOpt(
    MetModel = model_iJO1366_antiEpEX_scFv,
    medium = medium,
    bounds = bounds,
    costs = costs,
    opt_objective = opt_objective,
    biomass_objective = biomass_rxn_id,
    production_objective = protein_rxn_id,
    n_start = n_start, 
    n_iter = n_iter,
    n_candidates = n_candidates,
    model_objective = combined_objective,
    start_time = start_time,
    medium_linear_equality_constraints = None,
    medium_linear_inequality_constraints = None,
    medium_nonlinear_inequality_constraints = None,
    use_pfba = False
    )

profiler6.disable()
profiler6.dump_stats("profile_iJO1366_gpc_fixedM9.prof")

# print the 30 most expensive functions
stats6 = pstats.Stats(profiler6)
stats6.sort_stats("cumtime").print_stats(30)

# Display runtime
print_runtime(start_time)

# plot & save results
basename = (date + "_BayesOpt_iJO1366_antiEpEX_scFv_" + opt_objective + "_fixedM9_" + iterations + "_pfba")

# store results in JSON file
JSON_serialize_store_results(results_iJO1366_gpc, basename)

# plot all results
plot_pareto_batch_colour(
    results_iJO1366_gpc,
    xax = "growth rate tensors", 
    yax = "cost tensors",
    figname = (basename + "_pareto_growth-cost.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
plot_pareto_batch_colour(
    results_iJO1366_gpc,
    xax = "growth rate tensors", 
    yax = "production tensors",
    figname = (basename + "_pareto_growth-production.png"),
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
    
plot_3D(
    results_iJO1366_gpc,
    xax = "growth rate tensors", 
    yax = "cost tensors", 
    zax = "production tensors",
    figname = (basename + "_3D.png"),
    MetModel = model_iJO1366_antiEpEX_scFv, 
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )

# plot subset of results where production > 0.01 and growth > 0.5
plot_production_per_cost_coloured_by_growth(
    results_iJO1366_gpc,
    figname = (basename + "_best_coloured-by-growth.png"),
    growth_threshold = 0.5,
    production_threshold = 0.01,
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )
# plot subset of results where production > 0.01 and growth doesn't matter
plot_production_per_cost_coloured_by_growth(
    results_iJO1366_gpc,
    figname = (basename + "_best-production_coloured-by-growth.png"),
    growth_threshold = 0.0,
    production_threshold = 0.01,
    MetModel = model_iJO1366_antiEpEX_scFv,
    initial_medium = medium,
    initial_costs = costs,
    model_objective = combined_objective
    )

# How to load and deserialise the results
#result = JSON_deserialize_load_results((basename + ".json"), model_iJO1366_antiEpEX_scFv)